# MeKi Hybrid Memory — Al-Nisyan Upgrade
**Goal**: Test MeKi + LM2 + kNN + Episodic memory hybrid on Kaggle T4

**Requirements**: GPU T4 enabled, Internet ON

**Expected VRAM**: ~4-5GB (Qwen3.5-0.6B + Hybrid modules)

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth transformers accelerate bitsandbytes torch

In [ ]:
# Cell 2: Clone repo and setup path
import os, sys
repo_path = '/kaggle/working/al-nisyan'
if not os.path.exists(repo_path):
    !git clone https://github.com/faresrafat3/al-nisyan.git {repo_path}
sys.path.insert(0, repo_path)
os.chdir(repo_path)
print('Repo ready at:', repo_path)

In [ ]:
# Cell 3: Upload meki_hybrid.py (or copy from output)
# Option A: If meki_hybrid.py is uploaded as Kaggle Input
import shutil
from pathlib import Path

meki_source = None
for p in Path('/kaggle/input').rglob('meki_hybrid.py'):
    meki_source = p
    break

if meki_source:
    shutil.copy(meki_source, f'{repo_path}/src/models/meki_hybrid.py')
    print(f'Copied from: {meki_source}')
else:
    # Option B: Write directly (paste content here if needed)
    print('Please upload meki_hybrid.py as Kaggle Input, or paste the code')

In [ ]:
# Cell 4: Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# Cell 5: Load base model (0.6B for T4)
from src.models.memory_augmented_model_v3 import CultivatedMemoryModelV3

base = CultivatedMemoryModelV3(
    base_model_key='Qwen/Qwen3.5-0.6B',  # 0.6B = ~2GB VRAM
    memory_slots=64,
    memory_dim=512,
    controller_mode='clean',
)

model = base.model
tokenizer = base.tokenizer

print(f'Base model loaded. VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

In [ ]:
# Cell 6: Wrap with MeKi Hybrid Injector
from src.models.meki_hybrid import MeKiHybridInjector, MeKiBenchmark

injector = MeKiHybridInjector(
    model,
    tokenizer,
    meki_mem_dim=64,       # Reduced for T4
    lm2_mem_slots=128,      # Reduced for T4
    knn_max_size=4096,      # Reduced for T4
    knn_k=8,
    enable_meki=True,
    enable_lm2=True,
    enable_knn=True,
)

print(f'Hybrid ready. VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

In [ ]:
# Cell 7: Quick test
print('=== Test 1: Simple math ===')
r1 = injector.generate('What is 5 + 3?', max_new_tokens=20)
print(f'Q: What is 5 + 3?\nA: {r1}\n')

print('=== Test 2: Science ===')
r2 = injector.generate('What is photosynthesis?', max_new_tokens=40)
print(f'Q: What is photosynthesis?\nA: {r2}\n')

print('=== Test 3: Repeat (should use memory) ===')
r3 = injector.generate('What is 5 + 3?', max_new_tokens=20)
print(f'Q: What is 5 + 3? (2nd time)\nA: {r3}')

In [ ]:
# Cell 8: Full Benchmark
math_qs = [
    'What is 5 + 3?',
    'Calculate 10 times 2.',
    'What is 15 minus 7?',
    'What is 8 divided by 2?',
    'Calculate 3 plus 3.',
    'What is 100 plus 200?',
    'Calculate 7 times 8.',
    'What is 50 minus 25?',
    'Calculate 6 times 7.',
    'What is 2 to the power of 10?',
]

recall_qs = [
    'What is 5 + 3?',          # Repeat - should use memory
    'Calculate 10 times 2.',    # Repeat
    'What is the capital of France?',  # New
    'Calculate 7 times 8.',     # Repeat
    'Who wrote Romeo and Juliet?',  # New
]

benchmark = MeKiBenchmark(injector)
benchmark.run_phase1_learning(math_qs)
benchmark.run_phase2_recall(recall_qs)
results = benchmark.report()

In [ ]:
# Cell 9: Save results
import json

output = {
    'latency_base': results['base']['times'],
    'latency_hybrid': results['hybrid']['times'],
    'novelties': results['hybrid']['novelties'],
    'vram_gb': torch.cuda.memory_allocated() / 1024**3,
}

with open('/kaggle/working/meki_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Results saved to /kaggle/working/meki_results.json')
print('\nDownload the file from Output panel on the right')